---
# <div style="text-align: center"> Introduction </div>
---

Along these tutorials, we will see how <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System** class and its sources: the **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**
6) Running <span style="color:blue">**SCOPE**</span> - Part 1: File Structure
7) Running <span style="color:blue">**SCOPE**</span> - Part 2: Execution 
8) Running <span style="color:blue">**SCOPE**</span> - Part 3: Detailed Actions

---
# <div style="text-align: center"> Tutorial 1: Systems and their Sources</div>
---

In this tutorial, you will discover one of the main pillars of SCOPE: the **System**-class:

A **System** is a collection of chemical entities with something in common, typically its chemical composition. The Main Purpose of this class is to store paths for later use in SCOPE, as well as its **sources**.

- A **Source** is one of those chemical entities. There are two main types of sources: the **Cell** and the **Specie**:
- A **Specie** is an isolated chemical specie. It has various subclasses: being **Molecule** and **Ligand** the most important  
- A **Cell** is a periodic structure. Basically, a Specie that includes cell parameters 

 
In the first part of this tutorial, we will open an existing **SYSTEM**, stored in a binary file, and navigate the hierarchy of chemical species that it contains.  
In the second part, we will create a new system from scratch, and feed it with **sources**

## Part 1. Navigate an existing SYSTEM

In [ ]:
import os
from scope.read_write import load_binary

In [ ]:
## Path of the data folder contained in this tutorial. It should be "os.path.abspath('.')+'/Data/Tutorial_1/"
data_folder = os.path.abspath('.')+'/Data/1-Tutorial_1/'
## Loads the System object from a binary file, provided in the tutorial folder
sys = load_binary(f"{data_folder}ABITEM.npy")

In [ ]:
## All objects in SCOPE have a __repr__ method, so printing shows a summary of the object
print(sys)

The System is called ABITEM, and is associated with 6 **sources**:

```python
     0: cell, ABITEM01, H104-C92-N24-O4-S8-Fe4 
     1: cell, ABITEM, H104-C92-N24-O4-S8-Fe4 
     2: cell, ref_hs_cell, H104-C92-N24-O4-S8-Fe4 
     3: cell, ref_ls_cell, H104-C92-N24-O4-S8-Fe4 
     4: specie, ref_hs_mol, H18-C20-N6-S2-Fe 
     5: specie, ref_ls_mol, H18-C20-N6-S2-Fe 
```

Sources 0-1 are the **Cell** objects imported from the cell2mol interpretation of the .cif files  
Sources 2-3 are the **Cell** objects identified as reference HS and LS unit cells.  
Sources 4-5 are the **Specie** objects identified as reference HS and LS molecules.   

Below, we will navigate these sources to discover the Cell and Specie classes

### Sources 1: The CELL Class


In [ ]:
#All objects in SCOPE can be printed, which displays their key information. 
cell = sys.sources[0]
print(cell)

In [ ]:
# Have a look at the CELL object above.

# -It has 236 atoms, whose coordinates are stored as cell.coord, and labels as cell.labels.
print("Labels and Coordinates:", cell.labels[0], cell.coord[0]) 
# -It has cell parameters, stored as cell.cell_param
print("Cell Parameters:", cell.cell_param)
# -It has 8 Molecules, stored inside cell.moleclist
print("Number of Molecules:", len(cell.moleclist))
# -It has 2 unique (i.e. types of) molecules, with formulae:
for mol in cell.refmoleclist:
    print("Unique Molecule Formula:", mol.formula)

In [ ]:
# Apart from attributes, you can also run CELL-class functions.
# Here we try cell.check_fragmentation(), which checks if any molecule appears fragmented in the unit cell due to how the cell is cut
cell.check_fragmentation()

In [ ]:
# This CELL is a little bit special, though. It was created by the "Spin_Crossover" add-on included in SCOPE. 
# This add-on creates SCO_CELLs, which are subclasses of CELL, created to accomodate the specificities of this type of molecules.
# Notice the subtype
print(cell.type)
print(cell.subtype)

In [ ]:
# This CELL was created from cell2mol data, and is associated with a CIF file, whose information is stored in a Cif-class object: 
print(cell.cif)

In [ ]:
# You can view the unit cell with cell.view(), which opens a 3D viewer
cell.view(size='large')

### Sources 2: The SPECIE Class

In [ ]:
# Hopefully, you can identify the 4 transition metal complexes (TMCs) and the 4 solvent molecules in the viewer above.
# In any case, you can access those molecules through the cell.moleclist attribute. 
len(cell.moleclist)

#### Molecule Sub-Class

In [ ]:
# MOLECULE-class objects are a subclass of SPECIE, which more broadly refers to any chemical specie, including ligands in transition metal complexes
# Here, we take the first molecule in the unit cell as an example:
mol = cell.moleclist[0]
# Again, the key information can be displayed by printing the object
print(mol)


In [ ]:
# Notice the molecule is of type=SPECIE, and subtype=MOLECULE. As mentioned above, MOLECULE is a subclass of SPECIE. 
# Also, this molecule is a TMC, as shown by:
print(mol.iscomplex)
# SCOPE will flag a molecule as a TMC whenever it has any atom of the d- or f-blocks of the periodic table.
# Whenever this happens, it will automatically split the molecule into metals and ligands 

In [ ]:
# Ligands can be accessed with the "ligands" attribute: 
print(len(mol.ligands), "ligands in this TMC")

In [ ]:
# Similarly, Metals can be accessed with the "metals" attribute:
print(len(mol.metals), "metal atom(s) in this TMC")
# Metals belong to the ATOM class, which we will soon discover

In [ ]:
## All species can be visualized with the .view() method:
mol.view()

#### Ligand Sub-Class

In [ ]:
## Again, you can access the ligand objects through the "ligands" attribute
lig = mol.ligands[0]
print(lig)

In [ ]:
## The LIGAND is also a subclass of SPECIE (type=specie, subtype=ligand). 
print(lig.type, lig.subtype)
## It has some useful functions too, e.g.:
print(lig.get_denticity())        ## This one retrieves the denticity of the ligand
print(lig.get_connected_metals()) ## This one retreives all metals to which this ligand is connected 

In [ ]:
## As mentioned above, the Cell objects in this tutorial were imported from cell2mol CELL objects. 
## In these Cells, Ligands are associated with an rdkit object...
lig.rdkit_obj  

In [ ]:
## ... and a Smiles
lig.smiles

#### Metal Sub-Class

In [ ]:
## Again, METAL is a subclass of ATOM (type=atom, subtype=metal)
## You can access the METAL object from the ligand with (1) lig.get_connected_metals(), as we've seen before, or (2) From the molecule: 
met = mol.metals[0]
print(met)

In [ ]:
## Apart from attributes, it also has useful functions, e.g.:
met.get_coord_sphere()          ## These are the atoms that are coordinated to the metal
met.get_coord_sphere_formula()  ## This is its formula

In [ ]:
#met.get_cshm()   ## This one retrieves the Coordination Shape Measure of the metal.
                  ## The CShM quantifies how close the coordination environment is to an ideal geometry, which is an octahedron by default.
                  ## Unfortunately, the package that computes it (Cosymlib) introduces dependency issues during the installation of SCOPE 
                  ## Those dependencies make the installation of Scope a challenge
                  ## If you're skilled enough, you can try to install it manually

#### Going Upwards: Parents

In [ ]:
## So far, we went down in the hierarchy: System -> Cell -> Molecule -> Metal/Ligand. You can also go back up:

# This retrieves the parent CELL object of the ligand
parent = lig.get_parent('cell') 
print(parent)

In [ ]:
# And this retrieves the index of the ligand atoms in the parent molecule, so you can easily locate the child's atoms in parent attributes
idx = lig.get_parent_indices('molecule') 
print("Ligand Atom Indices in Parent's Molecule class:", idx)

for i, pi in enumerate(idx):
    lig_atom = lig.atoms[i]                                             ## i is the index in ligand atoms: 0,1,2 
    mol_atom = lig.get_parent('molecule').atoms[pi]                     ## pi is the index of the same atom in the parent molecule's atoms: 2,7,41
    print(i, pi, lig_atom.label, mol_atom.label, lig_atom == mol_atom)  ## Here, it compares both atoms. If TRUE, they are the same

#### The Elephant in the Room: The ATOM Class

In [ ]:
# As we've briefly seen above, species have atoms. 
len(mol.atoms)

In [ ]:
## The first atom of the molecule happens to be the metal. The second is a regular atom. Notice the different subtype
for at in mol.atoms[0:10]:                                  ## Printing the first 10 atoms
    print(at.get_parent_index('molecule'), at.label, at.subtype, at.charge, at.madjnum, at.adjnum)

## at.charge stores the formal atomic charge.
## at.madjnum is the number of adjacencies the atom has with a metal center.
## at.adjnum is the total number of adjacencies the atom has with any other atom

#### Charge and Spin

In [ ]:
## Because cell2mol infers the charge of species in a unit cell, this info is also stored in the molecules:
for mol in cell.moleclist:
    print(f"Formula: {mol.formula}, Charge: {mol.charge}, Spin: {mol.spin}")

In [ ]:
## And also for the ligands and metals inside the molecules:
mol = cell.moleclist[0]
for lig in mol.ligands:
    print(f"Ligand Formula: {lig.formula}, Charge: {lig.charge}, Spin: {lig.spin}")
for met in mol.metals:
    print(f"Metal: {met.label}, Charge: {met.charge}, Spin: {met.spin}")

In [ ]:
## All the way down to the ATOM objects, which are the actual bearers of spin and charge:
for at in mol.atoms:
    print(f"Atom: {at.label}, Charge: {at.charge}, Spin: {at.spin}")

In [ ]:
## IMPORTANT: The charge and spin attribute for the SPECIE (molecules, ligands) and CELL classes are computed as the sum of the charge and spin of their constituent ATOM objects.mol
## - By "spin" we mean the total number of unpaired electrons, i.e. a spin of 1 means one unpaired electron (doublet), a spin of 2 means two unpaired electrons (triplet), etc.
## - If you want to modify their charge, you should modify the charge/spin of an atom object directly: eg: at.set_charge(1), at.set_spin(2), where "at" is an ATOM object.
##   Alternatively, you can use spec.set_atomic_charges([list_of_charges]) and spec.set_atomic_spins([list_of_spins]) to set all atomic charges/spins of a SPECIE (spec) at once.

## Here we take atom 0, which should be an Fe atom,
at = mol.atoms[0]

## We set it as a high-spin center, with 4 unpaired electrons (spin=4)
at.set_spin(4)

## When introducing the spin, the multiplicity and ms values are also stored as attributes:
print(f"Spin: {at.spin}, Ms: {at.ms}, Multiplicity: {at.multiplicity}")

## When Working with the STATE class in tutorial number 2, we will see how we can rapidly make changes to the charge and spin of specific atoms (typically metals) in the state. 

#### Bond Class

In [ ]:
## Bonds are also stored as objects, with very limited functionality for now:
print(at.bonds)
## The formal bond order (bond.order) value above is extracted from the rdkit object, if available

## Part 2. Create a System
---

We will now create a simple system, that we will use later during Tutorial 7.

In [ ]:
## Path of Tutorial 7's Data folder. Normally, it should be "os.path.abspath('.')+'/Data/Tutorial_7/". Change if necessary
tutorial_folder = os.path.abspath('.')+'/Data/7-Tutorial_7/'

In [ ]:
from scope.classes_system import System
## To create a system, you only need a name
test_sys   = System("water")
print(test_sys)

In [ ]:
## The created system is still 'empty'. That is, no sources are associated to it yet. 
## Before feeding it with sources, we will set the paths of the three relevant folders:
## - Sources         (test_sys.sources_path)
## - Computations    (test_sys.computations_path)
## - System          (test_sys.system_path)

In [ ]:
## You can do it in 3 ways:

## (1) Directly as: 
test_sys.sources_path      = f"{tutorial_folder}1-Sources/water/"
test_sys.system_path       = f"{tutorial_folder}2-Systems/water/"
test_sys.system_file       = f"{tutorial_folder}2-Systems/water/water.npy"
test_sys.computations_path = f"{tutorial_folder}3-Computations/water/"

## (2) Reading from Environment-class object (uncomment to use). If you're working on a notebook, it might be more difficult to use:
# First, you load (or create) the enviroment
#   env = load_binary(path) 
# Second:
#   self.set_paths_from_environment(env)

## (3) With self.set_paths(), and you just need to follow the instructions. If you're working on a notebook, it might be more difficult to use

In [ ]:
## Once done, you can print the system to check that everything is correct
print(test_sys)

## You can check whether the paths are correct with. It will also return TRUE/FALSE if folders exist: 
print(test_sys.check_paths(debug=1))


In [ ]:
## At this stage, WARNINGS might pop up because some folder might not exist if you used option (1) above. 
## Folders are automatically created with options (2) and (3). So don't worry for the moment

### Create a Molecule and Associate it to the System

#### The basics

In [ ]:
## Now we create an actual water molecule, whose geometry is read from an .xyz file
from scope.classes_specie import Molecule
from scope.read_write import read_xyz

labels, coords = read_xyz(test_sys.sources_path+"geom1.xyz")
molec          = Molecule(labels, coords)
print(molec)

#### Charge and Spin

In [ ]:
## By default, molecules are neutral (and singlets) when created. 
## You can change it if needed:
molec.set_total_charge(2)
## Notice that the charge has changed:
molec

In [ ]:
## molec.set_total_charge() and molec.set_total_spin() are enough if you just want to specify charge and multiplicity for a quantum chemistry computation.
## These functions just set the charge to the first atom of the molecule
molec.atoms[0] 

## Notice that we have here a H atom with +2 charge (!)

In [ ]:
## It is much better if you individually select the atom you want to put the charge in, like:
molec.reset_charge()
molec.atoms[1].set_charge(-1)
molec

## Notice that we reset the charge, and now added a -1 charge to the O atom, which makes slightly more sense. 
## SPECIE and CELL charges/spins are always computed as the sum of atomic charges/spins hosted in its ATOMS.
## This means that, if you change an ATOM charge/spin, you're automatically changing the charge of the parent class 

In [ ]:
## We reset the charge, since we're gonna use this molecule in Tutorial 7:
molec.reset_charge()

In [ ]:
## For SPIN, it works very similarly: 
molec.set_spin_metals(2, debug=1)

## Logically, it won't find any metal in water, and it will complain, but this is something that should work in a TMC
## Most functions have a debug variable as input, with 1 'being verbose', and 0 being 'quiet'. 

In [ ]:
## Fortunately, we still have one of ABITEM's molecules in memory. Here it should work
mol.set_spin_metals(2, debug=1)  ## 'mol' is one of ABITEM's molecules

#### Initial State (to be explained in Tutorial 3)

In [ ]:
## Now we Create The Initial State, from which we will draw the starting data for the computations.
ini_state = molec.add_state("initial")
## As we mention in Tutorial 3, "initial" states are not created automatically. The reason is that the information they require depends on the source type.
## SPECIE sources only require geometry: 
ini_state.set_geometry(labels, coords)
## CELL sources would also require the cell_vectors and/or cell_parameters, which we don't have here:
#ini_state.set_cell() 

### Add Molecule as Source, and Save the system

In [ ]:
## And source it to our system, giving it the name "geom1"
_ = test_sys.add_source('geom1',molec)
## Notice that the molecule has been added in the list of sources:
print(test_sys)

In [ ]:
## Trying to add new molecules with the same name won't work, to avoid duplicities
_ = test_sys.add_source('geom1',molec, debug=1)

In [ ]:
## Finally, save the system to a file. By default, it will use test_sys.sys_file
test_sys.save()